In [1]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv("../.env")

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "API key not found. Check your .env file."
    )

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

ModuleNotFoundError: No module named 'dotenv'

In [2]:
import sys

print(sys.executable)

c:\Users\jhabh\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [1]:
%pip install python-dotenv google-genai


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
from google import genai

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Find the .env file in the project folder
env_path = Path.cwd().resolve().parent / ".env"

load_dotenv(env_path)

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("API key not found. Check your .env file.")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [4]:
models = client.models.list()

available_models = []

for model in models:
    name = model.name or ""

    if "generateContent" in (model.supported_actions or []):
        available_models.append(name)

print("Models available for text generation:")

for model in available_models:
    print(model)

Models available for text generation:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-

In [6]:
import time
from google import genai

# Select the model for our experiment
MODEL_NAME = "gemini-3.8-flash"

# The question we want to test
question = """
A shopkeeper buys an item for ₹800 and sells it for ₹1000.
Calculate the profit percentage.
"""

# Method A: Direct-answer prompting
direct_prompt = f"""
Solve the following question.
Provide only the final answer.

Question: {question}
"""

# Method B: Structured reasoning prompting
structured_prompt = f"""
Solve the following question using a structured approach.
Identify the relevant quantities, perform the calculations,
verify the result, and provide the final answer.

Question: {question}
"""

# Run Method A
start_time = time.perf_counter()

direct_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=direct_prompt
)

direct_time = time.perf_counter() - start_time

# Run Method B
start_time = time.perf_counter()

structured_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=structured_prompt
)

structured_time = time.perf_counter() - start_time

# Display the results
print("===== DIRECT ANSWER =====")
print(direct_response.text)
print(f"Response time: {direct_time:.2f} seconds")

print("\n===== STRUCTURED REASONING =====")
print(structured_response.text)
print(f"Response time: {structured_time:.2f} seconds")

===== DIRECT ANSWER =====
25%
Response time: 2.78 seconds

===== STRUCTURED REASONING =====
Here is the step-by-step solution to the problem:

### **1. Identify the Relevant Quantities**
* **Cost Price ($CP$):** The price at which the item was bought = ₹800
* **Selling Price ($SP$):** The price at which the item was sold = ₹1000

---

### **2. Perform the Calculations**

* **Step A: Calculate the Profit**
  $$\text{Profit} = SP - CP$$
  $$\text{Profit} = 1000 - 800 = ₹200$$

* **Step B: Calculate the Profit Percentage**
  $$\text{Profit Percentage} = \left(\frac{\text{Profit}}{CP}\right) \times 100$$
  $$\text{Profit Percentage} = \left(\frac{200}{800}\right) \times 100$$
  $$\text{Profit Percentage} = \frac{1}{4} \times 100 = 25\%$$

---

### **3. Verify the Result**
* Calculate $25\%$ of the Cost Price ($₹800$):
  $$25\% \text{ of } 800 = 0.25 \times 800 = ₹200$$
* Add the profit to the Cost Price to check if it equals the Selling Price:
  $$\text{Selling Price} = CP + \text{Profit} 

In [7]:
import time
import pandas as pd
from pathlib import Path

# Use the model that worked previously
MODEL_NAME = "gemini-3.8-flash"

# Our test question
question = (
    "A shopkeeper buys an item for ₹800 and sells it for ₹1000. "
    "Calculate the profit percentage."
)

# Define the two prompting methods
prompts = {
    "Direct": f"""
Solve the following question.
Provide only the final answer.

Question: {question}
""",

    "Structured": f"""
Solve the following question using a structured approach.
Identify the relevant quantities, perform the calculations,
verify the result, and provide the final answer.

Question: {question}
"""
}

# Run both methods and collect results
results = []

for method, prompt in prompts.items():

    print(f"Running {method} prompting...")

    start_time = time.perf_counter()

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    elapsed_time = time.perf_counter() - start_time

    answer = response.text or ""

    # Get token usage when provided by the API
    usage = response.usage_metadata

    input_tokens = (
        getattr(usage, "prompt_token_count", None)
        if usage else None
    )

    output_tokens = (
        getattr(usage, "candidates_token_count", None)
        if usage else None
    )

    results.append({
        "question_id": "Q001",
        "category": "Mathematics",
        "difficulty": "Easy",
        "method": method,
        "question": question,
        "response": answer,
        "response_time_seconds": round(elapsed_time, 3),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens
    })

# Display results
results_df = pd.DataFrame(results)

display(results_df)

Running Direct prompting...
Running Structured prompting...


,question_id,category,difficulty,method,question,response,response_time_seconds,input_tokens,output_tokens
0,Q001,Mathematics,Easy,Direct,A shopkeeper buys an item for ₹800 and sells i...,25%,145.683,44,3
1,Q001,Mathematics,Easy,Structured,A shopkeeper buys an item for ₹800 and sells i...,Here is the step-by-step solution using the st...,3.867,62,354


In [8]:
# Find the project folder
project_dir = Path.cwd().resolve().parent

# Create the results directory if necessary
results_dir = project_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# Save the experiment
output_path = results_dir / "pilot_results.csv"

results_df.to_csv(output_path, index=False)

print("Pilot experiment saved successfully!")
print(f"File location: {output_path}")

Pilot experiment saved successfully!
File location: C:\Users\jhabh\OneDrive\Desktop\LLM_Reasoning_Experiment\results\pilot_results.csv


In [9]:
print(results_df[[
    "method",
    "response_time_seconds",
    "input_tokens",
    "output_tokens"
]])

       method  response_time_seconds  input_tokens  output_tokens
0      Direct                145.683            44              3
1  Structured                  3.867            62            354


In [10]:
import pandas as pd
from pathlib import Path

# Locate the project directory
project_dir = Path.cwd().resolve().parent

# Locate the dataset
dataset_path = project_dir / "data" / "questions.csv"

# Load the CSV
questions_df = pd.read_csv(dataset_path)

# Display the dataset
display(questions_df)

,id,category,difficulty,question,correct_answer
0,Q001,Mathematics,Easy,What is 15 percent of 200?,30
1,Q002,Mathematics,Medium,An item costs 800 rupees and sells for 1000 ru...,25%
2,Q003,Mathematics,Hard,A number is increased by 20 percent and then d...,4% decrease
3,Q004,Logical,Easy,"A is taller than B, and B is taller than C. Wh...",C
4,Q005,Logical,Medium,All roses are flowers. No flowers are made of ...,No
5,Q006,Logical,Hard,Exactly one of A and B is telling the truth. A...,Yes
6,Q007,Aptitude,Easy,A vehicle travels 150 km in 3 hours at a const...,50 km/h
7,Q008,Aptitude,Medium,A can complete a job in 6 days and B can compl...,2 days
8,Q009,Aptitude,Hard,A 30-litre mixture contains 40 percent milk. H...,6 litres
9,Q010,Algorithms,Easy,Which data structure follows the Last In First...,Stack


In [11]:
# Check the dataset dimensions
print("Dataset shape:", questions_df.shape)

# Check the column names
print("\nColumns:")
print(questions_df.columns.tolist())

# Count questions in each category
print("\nQuestions by category:")
print(questions_df["category"].value_counts())

# Count questions by difficulty
print("\nQuestions by difficulty:")
print(questions_df["difficulty"].value_counts())

# Check for missing values
print("\nMissing values:")
print(questions_df.isnull().sum())

# Check for duplicate question IDs
print("\nDuplicate IDs:", questions_df["id"].duplicated().sum())

Dataset shape: (12, 5)

Columns:
['id', 'category', 'difficulty', 'question', 'correct_answer']

Questions by category:
category
Mathematics    3
Logical        3
Aptitude       3
Algorithms     3
Name: count, dtype: int64

Questions by difficulty:
difficulty
Easy      4
Medium    4
Hard      4
Name: count, dtype: int64

Missing values:
id                0
category          0
difficulty        0
question          0
correct_answer    0
dtype: int64

Duplicate IDs: 0


In [13]:
import os
import sys
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Locate the project directory
project_dir = Path.cwd().resolve().parent

# Add the project directory to Python's search path
sys.path.insert(0, str(project_dir))

# Now import the function from model.py
from src.model import generate_response

print("Project directory:", project_dir)
print("Model imported successfully!")


# Load the API key
load_dotenv(project_dir / ".env")

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("API key not found. Check your .env file.")

# Initialize Gemini
client = genai.Client(api_key=api_key)

# Load the dataset
dataset_path = project_dir / "data" / "questions.csv"
questions_df = pd.read_csv(dataset_path)

# Store experiment results
experiment_results = []

# Run both prompting methods on every question
for _, row in questions_df.iterrows():

    question_id = row["id"]
    question = row["question"]

    for method in ["Direct", "Structured"]:

        print(f"Question: {question_id} | Method: {method}")

        result = generate_response(
            client=client,
            question=question,
            method=method
        )

        experiment_results.append({
            "id": question_id,
            "category": row["category"],
            "difficulty": row["difficulty"],
            "method": method,
            "question": question,
            "correct_answer": row["correct_answer"],
            "response": result["response"],
            "response_time_seconds": result["response_time_seconds"],
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"]
        })

# Convert the results into a DataFrame
results_df = pd.DataFrame(experiment_results)

display(results_df.head())

Project directory: C:\Users\jhabh\OneDrive\Desktop\LLM_Reasoning_Experiment
Model imported successfully!
Question: Q001 | Method: Direct


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [14]:
import importlib
import src.model

importlib.reload(src.model)

from src.model import generate_response

print("Updated generate_response function loaded!")

Updated generate_response function loaded!


In [15]:
test_question = "What is 15 percent of 200?"

test_result = generate_response(
    client=client,
    question=test_question,
    method="Direct"
)

print("Response:", test_result["response"])
print("Response time:", test_result["response_time_seconds"], "seconds")
print("Input tokens:", test_result["input_tokens"])
print("Output tokens:", test_result["output_tokens"])

Temporary API error. Retrying in 2 seconds (attempt 1/4)...
Response: 30
Response time: 4.135 seconds
Input tokens: 30
Output tokens: 2


In [16]:
import importlib
import src.model

importlib.reload(src.model)

from src.model import generate_response

print("Updated function loaded successfully!")

Updated function loaded successfully!


In [17]:
test_result = generate_response(
    client=client,
    question="What is 15 percent of 200?",
    method="Direct"
)

print("Response:", test_result["response"])
print(
    "Successful request time:",
    test_result["response_time_seconds"],
    "seconds"
)
print(
    "Total elapsed time:",
    test_result["total_elapsed_seconds"],
    "seconds"
)
print("Input tokens:", test_result["input_tokens"])
print("Output tokens:", test_result["output_tokens"])

Response: 30
Successful request time: 5.764 seconds
Total elapsed time: 5.764 seconds
Input tokens: 30
Output tokens: 2


In [18]:
print("Response:", test_result["response"])
print("Successful request time:", test_result["response_time_seconds"], "seconds")
print("Total elapsed time:", test_result["total_elapsed_seconds"], "seconds")
print("Input tokens:", test_result["input_tokens"])
print("Output tokens:", test_result["output_tokens"])

Response: 30
Successful request time: 5.764 seconds
Total elapsed time: 5.764 seconds
Input tokens: 30
Output tokens: 2


In [19]:
# Reload the updated dataset
questions_df = pd.read_csv(dataset_path)

# Check the number of questions
print("Total questions:", len(questions_df))

# Check the categories
print("\nQuestions per category:")
print(questions_df["category"].value_counts())

# Check for missing values
print("\nMissing values:")
print(questions_df.isnull().sum())

# Verify Q006
print("\nCorrected Q006:")
display(questions_df[questions_df["id"] == "Q006"])

Total questions: 12

Questions per category:
category
Mathematics    3
Logical        3
Aptitude       3
Algorithms     3
Name: count, dtype: int64

Missing values:
id                0
category          0
difficulty        0
question          0
correct_answer    0
dtype: int64

Corrected Q006:


,id,category,difficulty,question,correct_answer
5,Q006,Logical,Hard,A three-digit code uses distinct digits. The f...,426


In [ ]:
# Store experiment results
experiment_results = []


# Run both prompting methods on every question
for _, row in questions_df.iterrows():

    question_id = row["id"]
    question = row["question"]

    for method in ["Direct", "Structured"]:

        print(f"\nQuestion: {question_id} | Method: {method}")

        try:
            result = generate_response(
                client=client,
                question=question,
                method=method
            )

            experiment_results.append({
                "id": question_id,
                "category": row["category"],
                "difficulty": row["difficulty"],
                "method": method,
                "question": question,
                "correct_answer": row["correct_answer"],
                "response": result["response"],
                "response_time_seconds": result["response_time_seconds"],
                "total_elapsed_seconds": result["total_elapsed_seconds"],
                "input_tokens": result["input_tokens"],
                "output_tokens": result["output_tokens"],
                "status": "Success"
            })

            print("Response:", result["response"])
            print("Successful request time:",
                  result["response_time_seconds"], "seconds")
            print("Total elapsed time:",
                  result["total_elapsed_seconds"], "seconds")
            print("Output tokens:", result["output_tokens"])

        except Exception as error:

            print("Request failed:", error)

            experiment_results.append({
                "id": question_id,
                "category": row["category"],
                "difficulty": row["difficulty"],
                "method": method,
                "question": question,
                "correct_answer": row["correct_answer"],
                "response": "",
                "response_time_seconds": None,
                "total_elapsed_seconds": None,
                "input_tokens": None,
                "output_tokens": None,
                "status": "Failed"
            })

# Convert results to a DataFrame
results_df = pd.DataFrame(experiment_results)

print("\nExperiment run finished!")
print("Total recorded attempts:", len(results_df))
print("Successful responses:", (results_df["status"] == "Success").sum())
print("Failed responses:", (results_df["status"] == "Failed").sum())

display(results_df.head())


Question: Q001 | Method: Direct
Response: 30
Successful request time: 3.763 seconds
Total elapsed time: 3.763 seconds
Output tokens: 2

Question: Q001 | Method: Structured
Response: Here is the step-by-step solution to the problem:

### **Step 1: Understand the terms and convert the percentage**
* "Percent" means "per one hundred." 
* Convert $15\%$ into a fraction or a decimal:
  $$\text{Fraction: } \frac{15}{100}$$
  $$\text{Decimal: } 0.15$$

---

### **Step 2: Perform the calculation**
To find $15\%$ of $200$, multiply the converted percentage by $200$:

* **Using fractions:**
  $$\frac{15}{100} \times 200 = 15 \times \frac{200}{100} = 15 \times 2 = 30$$

* **Using decimals:**
  $$0.15 \times 200 = 30$$

---

### **Step 3: Check the result**
We can check the answer using mental benchmark percentages:
* $10\%$ of $200 = 20$
* $5\%$ of $200 = 10$ (half of $10\%$)
* $15\% = 10\% + 5\% = 20 + 10 = 30$

The calculation is confirmed to be correct.

---

### **Final Answer:**
$15$ percen

,id,category,difficulty,method,question,correct_answer,response,response_time_seconds,total_elapsed_seconds,input_tokens,output_tokens,status
0,Q001,Mathematics,Easy,Direct,What is 15 percent of 200?,30,30,3.763,3.763,30.0,2.0,Success
1,Q001,Mathematics,Easy,Structured,What is 15 percent of 200?,30,Here is the step-by-step solution to the probl...,14.732,14.732,53.0,327.0,Success
2,Q002,Mathematics,Medium,Direct,An item costs 800 rupees and sells for 1000 ru...,25%,25%,2.110,2.110,41.0,3.0,Success
3,Q002,Mathematics,Medium,Structured,An item costs 800 rupees and sells for 1000 ru...,25%,Here is the step-by-step solution to the probl...,4.167,4.167,64.0,378.0,Success
4,Q003,Mathematics,Hard,Direct,A number is increased by 20 percent and then d...,4% decrease,4 percent decrease (or -4%),2.271,2.271,43.0,8.0,Success


In [21]:
print("Total rows:", len(results_df))

print("\nResults by method and status:")
display(
    results_df.groupby(["method", "status"]).size().unstack(fill_value=0)
)

print("\nFailed requests:")
display(
    results_df.loc[
        results_df["status"] == "Failed",
        ["id", "method", "response"]
    ]
)

Total rows: 24

Results by method and status:


status,Failed,Success
method,,
Direct,6,6
Structured,5,7



Failed requests:


,id,method,response
12,Q007,Direct,
14,Q008,Direct,
15,Q008,Structured,
16,Q009,Direct,
17,Q009,Structured,
18,Q010,Direct,
19,Q010,Structured,
20,Q011,Direct,
21,Q011,Structured,
22,Q012,Direct,


In [22]:
results_path = project_dir / "results" / "results_partial.csv"

results_df.to_csv(results_path, index=False)

print("Partial results saved to:")
print(results_path)

Partial results saved to:
C:\Users\jhabh\OneDrive\Desktop\LLM_Reasoning_Experiment\results\results_partial.csv


In [23]:
import time
import pandas as pd

# Keep a copy of the existing results
recovered_results = results_df.copy()

# Identify the failed requests
failed_requests = recovered_results[
    recovered_results["status"] == "Failed"
].copy()

print("Failed requests to retry:", len(failed_requests))

# Retry only failed requests
for index, row in failed_requests.iterrows():

    print(f"\nRetrying {row['id']} | {row['method']}")

    try:
        result = generate_response(
            client=client,
            question=row["question"],
            method=row["method"]
        )

        # Update the existing row with the new response
        recovered_results.loc[index, "response"] = result["response"]
        recovered_results.loc[index, "response_time_seconds"] = (
            result["response_time_seconds"]
        )
        recovered_results.loc[index, "total_elapsed_seconds"] = (
            result["total_elapsed_seconds"]
        )
        recovered_results.loc[index, "input_tokens"] = result["input_tokens"]
        recovered_results.loc[index, "output_tokens"] = result["output_tokens"]
        recovered_results.loc[index, "status"] = "Success"

        print("Retry successful!")
        print("Response:", result["response"])

    except Exception as error:

        # Keep the request marked as failed and display the actual error
        recovered_results.loc[index, "response"] = ""
        recovered_results.loc[index, "status"] = "Failed"

        print("Retry failed.")
        print("Error:", repr(error))

    # Short pause between requests
    time.sleep(3)

    # Save progress after every request
    results_path = project_dir / "results" / "results.csv"
    recovered_results.to_csv(results_path, index=False)

# Replace the old DataFrame with the recovered results
results_df = recovered_results

print("\nRecovery attempt completed.")
print("Successful:", (results_df["status"] == "Success").sum())
print("Failed:", (results_df["status"] == "Failed").sum())
print("Saved to:", results_path)

Failed requests to retry: 11

Retrying Q007 | Direct
Temporary API error. Retrying in 2 seconds (attempt 1/4)...
Temporary API error. Retrying in 4 seconds (attempt 2/4)...
Temporary API error. Retrying in 8 seconds (attempt 3/4)...
Retry failed.
Error: ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\\nPlease retry in 17h33m45.504150759s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/goo

In [24]:
results_path = project_dir / "results" / "results_partial.csv"

results_df.to_csv(results_path, index=False)

print("Results backed up successfully!")
print("Successful responses:", (results_df["status"] == "Success").sum())
print("Failed responses:", (results_df["status"] == "Failed").sum())
print("Saved to:", results_path)

Results backed up successfully!
Successful responses: 13
Failed responses: 11
Saved to: C:\Users\jhabh\OneDrive\Desktop\LLM_Reasoning_Experiment\results\results_partial.csv
